# Travail de groupe ETL

On commence par importer les librairies nécessaires et charger les fichiers CSV dans un dictionnaire de DataFrames. Ensuite, nous créons une copie du dictionnaire original pour conserver les données brutes.

In [ ]:
# Imports des bibliothèques

import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

all_files = sorted(glob.glob("*.csv"))

df_dic = {}
df_origin = {}
for file in all_files:
    nom = file.replace("datasets/Olist/", "").replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    df_dic[nom] = pd.read_csv(file)
    df_origin[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

## Aperçu des tables

In [6]:
for nom, df in df_dic.items():
    print("=====", nom, "=====")
    display(resume(df))

===== customers =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
customer_id,str,99441,ffffe8b65bbe3087b653a978c870db99,00012a2ce6f8dcda20d059ce98491703,0,0.0
customer_unique_id,str,96096,ffffd2657e2aad2907e67c3e9daecbeb,0000366f3b9a7992bf8c76cfdf3221e2,0,0.0
customer_zip_code_prefix,int64,14994,99990,1003,0,0.0
customer_city,str,4119,zortea,abadia dos dourados,0,0.0
customer_state,str,27,TO,AC,0,0.0


===== geolocation =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
geolocation_zip_code_prefix,int64,19015,99990,1001,0,0.0
geolocation_lat,float64,717372,45.065933,-36.605374,0,0.0
geolocation_lng,float64,717615,121.105394,-101.466766,0,0.0
geolocation_city,str,8011,óleo,* cidade,0,0.0
geolocation_state,str,27,TO,AC,0,0.0


===== order_items =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,98666,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
order_item_id,int64,21,21,1,0,0.0
product_id,str,32951,fffe9eeff12fcbd74a2f2b007dde0c58,00066f42aeeb9f3007548bb9d3f33c38,0,0.0
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
shipping_limit_date,str,93318,2020-04-09 22:35:08,2016-09-19 00:15:34,0,0.0
price,float64,5968,6735.0,0.85,0,0.0
freight_value,float64,6999,409.68,0.0,0,0.0


... (8 outputs left out)

===== sellers =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
seller_zip_code_prefix,int64,2246,99730,1001,0,0.0
seller_city,str,611,xaxim,04482255,0,0.0
seller_state,str,23,SP,AC,0,0.0


===== product_category_name_translation =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
product_category_name,str,71,utilidades_domesticas,agro_industria_e_comercio,0,0.0
product_category_name_english,str,71,watches_gifts,agro_industry_and_commerce,0,0.0


## 1 - Table Orders

In [19]:
orders = df_dic["orders"]

print(orders.describe().round(2))
print("\n")
print(resume(orders))

         order_purchase_timestamp           order_approved_at  \
count                       99441                       99441   
mean   2017-12-31 08:43:12.776581  2017-12-31 19:08:20.691654   
min           2016-09-04 21:15:19         2016-09-15 12:16:38   
25%           2017-09-12 14:46:19         2017-09-12 21:55:13   
50%           2018-01-18 23:04:36         2018-01-19 11:54:30   
75%           2018-05-04 15:42:16         2018-05-04 23:50:21   
max           2018-10-17 17:30:18  2018-10-18 03:46:54.361243   

      order_delivered_carrier_date order_delivered_customer_date  \
count                         1359                           275   
mean    2018-05-21 20:05:07.557607    2018-06-08 06:36:11.536170   
min     2017-02-25 12:34:24.066805    2017-03-05 22:28:38.521625   
25%     2018-04-26 19:41:03.566805    2018-05-05 02:02:45.521625   
50%     2018-07-03 05:26:55.066805    2018-07-14 20:49:49.521625   
75%     2018-07-08 10:32:55.066805    2018-07-16 01:43:03.021625   
max

### Conversion des colonnes en datetime

Les colonnes contenant des dates sont des `string` ce qui va empecher des opérations.

In [20]:
for col in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    orders[col] = pd.to_datetime(orders[col])

J'ai trouvé des incohérences de dates dans le fichier d'origine

In [21]:
dates_tests = [
    ("order_approved_at", "order_purchase_timestamp"),
    ("order_delivered_carrier_date", "order_approved_at"),
    ("order_delivered_customer_date", "order_delivered_carrier_date"),
]

for avant, apres in dates_tests:
    wrong_dates = (orders[avant] < orders[apres]).sum()
    print(f"Nombre de dates erronées pour {avant} < {apres} : {wrong_dates}")


Nombre de dates erronées pour order_approved_at < order_purchase_timestamp : 0
Nombre de dates erronées pour order_delivered_carrier_date < order_approved_at : 99
Nombre de dates erronées pour order_delivered_customer_date < order_delivered_carrier_date : 0


### Ajout d'une colonne imputee

Pour garder une trace des enregistrements modifiés on crée une colonne booléenne **imputee**

In [22]:
orders["imputee"] = (
    orders[
        [
            "order_delivered_customer_date",
            "order_delivered_carrier_date",
            "order_approved_at",
        ]
    ]
    .isna()
    .any(axis=1)
)

### Recherche des ecart moyens entres les colonnes

On va calculer les écarts seulements sur lignes avec le statut *delivered*

In [ ]:
ecart_approved = (
    (orders["order_approved_at"] - orders["order_purchase_timestamp"])
    .where(orders["order_status"] == "delivered")
    .mean()
)

ecart_delivered_carrier = (
    (orders["order_delivered_carrier_date"] - orders["order_approved_at"])
    .where(orders["order_status"] == "delivered")
    .where(orders["order_delivered_carrier_date"] > orders["order_approved_at"])
    .mean()
)

ecart_delivered_customer = (
    (orders["order_delivered_customer_date"] - orders["order_delivered_carrier_date"])
    .where(orders["order_status"] == "delivered")
    .where(
        orders["order_delivered_customer_date"] > orders["order_delivered_carrier_date"]
    )
    .mean()
)

ecart_customer_from_approved = (
    (orders["order_delivered_customer_date"] - orders["order_approved_at"])
    .where(orders["order_status"] == "delivered")
    .where(orders["order_delivered_customer_date"] > orders["order_approved_at"])
    .mean()
)

print(f"Écart entre l'approbation et l'achat : {ecart_approved}")
print(
    f"Écart entre l'approbation et la remise au transporteur : {ecart_delivered_carrier}"
)
print(
    f"Écart entre la remise au transporteur et la livraison au client : {ecart_delivered_customer}"
)
print(
    f"Écart entre l'approbation et la livraison au client : {ecart_customer_from_approved}"
)

### Remplacement des données manquantes par la moyenne

In [24]:
orders["order_approved_at"] = orders["order_approved_at"].fillna(
    orders["order_purchase_timestamp"] + ecart_approved
)

orders["order_delivered_carrier_date"] = orders["order_delivered_carrier_date"].fillna(
    orders["order_approved_at"] + ecart_delivered_carrier
)

orders["order_delivered_customer_date"] = orders[
    "order_delivered_customer_date"
].fillna(orders["order_delivered_carrier_date"] + ecart_delivered_customer)

### Correction incohérences des dates

In [37]:
orders["order_delivered_carrier_date"] = (
    orders["order_delivered_carrier_date"] + ecart_delivered_carrier
).where(orders["order_delivered_carrier_date"] < orders["order_approved_at"])

orders["order_delivered_customer_date"] = (
    orders["order_delivered_customer_date"] + ecart_delivered_customer
).where(
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
)

### Vérifier la cohérence des dates

Il faut vérifier que les dates créers soient cohérentes, pas de livraisons avant la validation.  
Pas d'incohérence supplémentaire par rapport au jeu d'origine.

In [38]:
dates_tests = [
    ("order_approved_at", "order_purchase_timestamp"),
    ("order_delivered_carrier_date", "order_approved_at"),
    ("order_delivered_customer_date", "order_delivered_carrier_date"),
]

for avant, apres in dates_tests:
    wrong_dates = (orders[avant] < orders[apres]).sum()
    print(f"Nombre de dates erronées pour {avant} < {apres} : {wrong_dates}")

print("\n")

for avant, apres in dates_tests:
    wrong_dates = (
        orders[avant].where(orders["imputee"] == True)
        < orders[apres].where(orders["imputee"] == True)
    ).sum()
    print(f"Nombre de dates imputées erronées pour {avant} < {apres} : {wrong_dates}")


Nombre de dates erronées pour order_approved_at < order_purchase_timestamp : 0
Nombre de dates erronées pour order_delivered_carrier_date < order_approved_at : 13
Nombre de dates erronées pour order_delivered_customer_date < order_delivered_carrier_date : 0


Nombre de dates imputées erronées pour order_approved_at < order_purchase_timestamp : 0
Nombre de dates imputées erronées pour order_delivered_carrier_date < order_approved_at : 12
Nombre de dates imputées erronées pour order_delivered_customer_date < order_delivered_carrier_date : 0


In [39]:
# cellule de visualisation des données
orders.describe().round(2)
resume(orders)

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,99441,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.00
customer_id,str,99441,ffffe8b65bbe3087b653a978c870db99,00012a2ce6f8dcda20d059ce98491703,0,0.00
order_status,str,8,unavailable,approved,0,0.00
order_purchase_timestamp,datetime64[us],98875,2018-10-17 17:30:18,2016-09-04 21:15:19,0,0.00
order_approved_at,datetime64[us],90893,2018-10-18 03:46:54.361243,2016-09-15 12:16:38,0,0.00
order_delivered_carrier_date,datetime64[us],27,2018-08-21 23:54:36.455003,2017-03-02 00:17:05.455003,99414,99.97
order_delivered_customer_date,datetime64[us],0,NaT,NaT,99441,100.00
order_estimated_delivery_date,datetime64[us],459,2018-11-12 00:00:00,2016-09-30 00:00:00,0,0.00
imputee,bool,2,True,False,0,0.00


In [ ]:
#PRODUCT COLUMN CLEANING


products = df_dic["products"]
col = products["product_photos_qty"]
col.fillna(0)

cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]


products["nb_identiques"] = products.groupby(cols, dropna=False)[cols[0]].transform("size") - 1

resultat = products.sort_values("nb_identiques", ascending=False)
print(resultat[["product_id", "product_category_name"] + cols + ["nb_identiques"]])


In [ ]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

combinaisons = products.value_counts(subset=cols, dropna=False)

print(combinaisons.head(1))
print(combinaisons.head(10))

# les combinaisons présentes au moins 2 fois
doublons = combinaisons[combinaisons > 1].index

# les lignes de products qui portent une de ces combinaisons
lignes_identiques = products[products.set_index(cols).index.isin(doublons)]

print(len(lignes_identiques), "produits partagent leurs 4 valeurs avec au moins un autre")

# répartition par catégorie (dropna=False fait apparaître les sans-catégorie, NaN)
repartition = lignes_identiques["product_category_name"].value_counts(dropna=False)
print(repartition)

# combien sont sans catégorie
print(lignes_identiques["product_category_name"].isna().sum(), "sont sans catégorie")
for i in range (10):
    print ("-")

sans_categorie = products[products["product_category_name"].isna()]

combinaisons1 = sans_categorie.value_counts(subset=cols, dropna=False)

print(combinaisons1.head(1))
print(combinaisons1.head(10))
total_identiques = combinaisons1[combinaisons1 > 1].sum()
print(total_identiques, "produits sans catégorie partagent leurs 4 valeurs avec au moins un autre")


In [ ]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

resultat = (products.groupby(cols, dropna=False)
            .agg(nb_lignes=("product_category_name", "size"),
                 categorie_majoritaire=("product_category_name",
                                         lambda s: s.value_counts(dropna=False).index[0]),
                 nb_categories=("product_category_name", "nunique"))
            .reset_index())

resultat = resultat[resultat["nb_lignes"] > 1].sort_values("nb_lignes", ascending=False)
resultat["nb_majoritaire"] = (products.groupby(cols, dropna=False)["product_category_name"]
                              .agg(lambda s: s.value_counts(dropna=False).iloc[0])
                              .values[resultat.index])

print(resultat.to_string(index=False))


In [ ]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

# prix par produit (au cas où pas encore fait)
order_items = df_dic['order_items']
prix_produit = order_items.groupby("product_id")["price"].median()
if "price" not in products.columns:
    products = products.merge(prix_produit.rename("price"), on="product_id", how="left")

g = products.groupby(cols, dropna=False)
cat_dans_groupe = g["product_category_name"].transform(lambda s: s.notna().sum())
nb_identiques   = g["product_category_name"].transform("size") - 1
prix_frere      = g["price"].transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None)

masque = (products["product_category_name"].isna()
          & products["product_name_lenght"].isna()
          & products["product_description_lenght"].isna()
          & (cat_dans_groupe == 0))

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(nb_identiques=nb_identiques[masque], prix=prix_frere[masque])
            .sort_values("nb_identiques", ascending=False))

print(len(resultat), "produits sans catégorie / nom / description et sans frère catégorisé")

affichage = (resultat.rename(columns={
                "product_id": "produit",
                "product_weight_g": "poids (g)",
                "product_length_cm": "long. (cm)",
                "product_height_cm": "haut. (cm)",
                "product_width_cm": "larg. (cm)",
                "nb_identiques": "nb frères"})
             .style
             .hide(axis="index")
             .format({"prix": "R$ {:.2f}",
                      "poids (g)": "{:.0f}", "long. (cm)": "{:.0f}",
                      "haut. (cm)": "{:.0f}", "larg. (cm)": "{:.0f}"},
                     na_rep="—")
             .background_gradient(subset=["nb frères"], cmap="Reds"))

display(affichage)


In [ ]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

# --- prix par produit (médiane des ventes) ---
order_items = df_dic['order_items']
prix_produit = order_items.groupby("product_id")["price"].median()
if "price" not in products.columns:
    products = products.merge(prix_produit.rename("price"), on="product_id", how="left")

# --- infos sur les frères (mêmes 4 dimensions) ---
g = products.groupby(cols, dropna=False)

cat_dans_groupe = g["product_category_name"].transform(lambda s: s.notna().sum())  # frères catégorisés
nb_categories   = g["product_category_name"].transform("nunique")                  # catégories différentes
nb_identiques   = g["product_category_name"].transform("size") - 1
categorie_majoritaire = g["product_category_name"].transform(
    lambda s: s.value_counts().index[0] if s.notna().any() else None)
nb_prix    = g["price"].transform("nunique")
prix_frere = g["price"].transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None)

# --- produits sans catégorie / nom / description, mais avec au moins un frère catégorisé ---
masque = (products["product_category_name"].isna()
          & products["product_name_lenght"].isna()
          & products["product_description_lenght"].isna()
          & (cat_dans_groupe > 0)
          & products[cols].notna().all(axis=1))   # écarte les produits sans dimensions

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(categorie_probable=categorie_majoritaire[masque],
                    nb_categories=nb_categories[masque],
                    nb_identiques=nb_identiques[masque],
                    nb_prix=nb_prix[masque],
                    price_match=(nb_prix[masque] == 1),
                    prix=prix_frere[masque])
            .sort_values(["nb_categories", "price_match", "nb_identiques"],
                         ascending=[True, False, False]))

print(len(resultat), "produits sans catégorie/nom/description avec frère catégorisé")
print((resultat["nb_categories"] == 1).sum(), "ont des frères tous d'accord sur une seule catégorie")
affichage = (resultat.rename(columns={
                "product_id": "produit",
                "product_weight_g": "poids (g)",
                "product_length_cm": "long. (cm)",
                "product_height_cm": "haut. (cm)",
                "product_width_cm": "larg. (cm)",
                "categorie_probable": "catégorie probable",
                "nb_categories": "nb catég.",
                "nb_identiques": "nb frères",
                "price_match": "prix idem"})
             .style
             .hide(axis="index")
             .format({"prix": "R$ {:.2f}",
                      "poids (g)": "{:.0f}", "long. (cm)": "{:.0f}",
                      "haut. (cm)": "{:.0f}", "larg. (cm)": "{:.0f}"})
             .background_gradient(subset=["nb frères"], cmap="Blues")
             .map(lambda v: "background-color:#d8f3dc" if v is True else "", subset=["prix idem"]))

display(affichage)


In [ ]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

nb_cat_groupe = (products.groupby(cols, dropna=False)["product_category_name"]
                 .transform("nunique"))

# nombre d'autres lignes avec les mêmes 4 valeurs
nb_identiques = (products.groupby(cols, dropna=False)["product_category_name"]
                 .transform("size") - 1)

categorie_frere = (products.groupby(cols, dropna=False)["product_category_name"]
                   .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None))

masque = (products["product_category_name"].isna() & (nb_cat_groupe == 1))

print(masque.sum(), "produits sans catégorie dont tous les frères sont du même groupe")

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(categorie_probable=categorie_frere[masque],
                    nb_identiques=nb_identiques[masque])
            .sort_values("nb_identiques", ascending=False))

print(resultat.to_string(index=False))


In [ ]:
cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

# --- 1. un prix par produit (médiane des ventes) ---
order_items = df_dic['order_items']
prix_produit = order_items.groupby("product_id")["price"].median()

if "price" not in products.columns:          # évite price_x / price_y si on relance
    products = products.merge(prix_produit.rename("price"), on="product_id", how="left")

# --- 2. infos sur les frères (lignes partageant les 4 mêmes dimensions) ---
g = products.groupby(cols, dropna=False)

nb_cat_groupe   = g["product_category_name"].transform("nunique")          # catégories différentes
nb_identiques   = g["product_category_name"].transform("size") - 1          # autres frères
categorie_frere = g["product_category_name"].transform(
                      lambda s: s.dropna().iloc[0] if s.notna().any() else None)
nb_prix         = g["price"].transform("nunique")
prix_frere      = g["price"].transform(
                      lambda s: s.dropna().iloc[0] if s.notna().any() else None)

# --- 3. produits sans catégorie, récupérables (frères unanimes + dimensions renseignées) ---
masque = (products["product_category_name"].isna()
          & (nb_cat_groupe == 1)
          & products[cols].notna().all(axis=1))

resultat = (products.loc[masque, ["product_id"] + cols]
            .assign(categorie_probable=categorie_frere[masque],
                    nb_identiques=nb_identiques[masque],
                    nb_prix=nb_prix[masque],
                    price_match=(nb_prix[masque] == 1),
                    prix=prix_frere[masque])
            .sort_values(["price_match", "nb_identiques"], ascending=[False, False]))

print(len(resultat), "produits récupérables")
print(resultat["price_match"].sum(), "confirmés aussi par le prix (price_match = True)")
print(resultat.to_string(index=False))


### Transformation de la table Order Items

In [41]:
items = df_dic["order_items"]

display(resume(items))

print("Dimensions :", items.shape)
print("Doublons :", items.duplicated().sum())

display(items.isna().sum())
display(items.describe())
display(items.dtypes)

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,98666,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
order_item_id,int64,21,21,1,0,0.0
product_id,str,32951,fffe9eeff12fcbd74a2f2b007dde0c58,00066f42aeeb9f3007548bb9d3f33c38,0,0.0
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
shipping_limit_date,datetime64[us],93318,2020-04-09 22:35:08,2016-09-19 00:15:34,0,0.0
price,float64,5968,6735.0,0.85,0,0.0
freight_value,float64,6999,409.68,0.0,0,0.0


Dimensions : (112650, 7)
Doublons : 0


order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

,order_item_id,shipping_limit_date,price,freight_value
count,112650.000000,112650,112650.000000,112650.000000
mean,1.197834,2018-01-07 15:36:52.192685,120.653739,19.990320
min,1.000000,2016-09-19 00:15:34,0.850000,0.000000
25%,1.000000,2017-09-20 20:57:27.500000,39.900000,13.080000
50%,1.000000,2018-01-26 13:59:35,74.990000,16.260000
75%,1.000000,2018-05-10 14:34:00.750000,134.900000,21.150000
max,21.000000,2020-04-09 22:35:08,6735.000000,409.680000
std,0.705124,NaN,183.633928,15.806405


order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

### Conversion de la colonne shipping_limit_date en datetime


In [42]:
items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"]
)

display(resume(items))

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,98666,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
order_item_id,int64,21,21,1,0,0.0
product_id,str,32951,fffe9eeff12fcbd74a2f2b007dde0c58,00066f42aeeb9f3007548bb9d3f33c38,0,0.0
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
shipping_limit_date,datetime64[us],93318,2020-04-09 22:35:08,2016-09-19 00:15:34,0,0.0
price,float64,5968,6735.0,0.85,0,0.0
freight_value,float64,6999,409.68,0.0,0,0.0


## Les livraisons gratuites ne doivent pas être considérées automatiquement comme des erreurs.

In [43]:
print("Prix négatifs :", (items["price"] < 0).sum())

print("Prix nuls :", (items["price"] == 0).sum())

print("Frais négatifs :", (items["freight_value"] < 0).sum())

print("Livraisons gratuites :", (items["freight_value"] == 0).sum())

Prix négatifs : 0
Prix nuls : 0
Frais négatifs : 0
Livraisons gratuites : 383


### Table Order Payments

In [44]:
payments = df_dic["order_payments"]

display(resume(payments))
print("Doublons exacts :", payments.duplicated().sum())

print("Clés dupliquées :", payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum())
display(payments["payment_type"].value_counts())

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,99440,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
payment_sequential,int64,29,29,1,0,0.0
payment_type,str,5,voucher,boleto,0,0.0
payment_installments,int64,24,24,0,0,0.0
payment_value,float64,29077,13664.08,0.0,0,0.0


Doublons exacts : 0
Clés dupliquées : 0


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [46]:
print("Paiements négatifs :",
      (payments["payment_value"] < 0).sum())

print("Paiements nuls :",
      (payments["payment_value"] == 0).sum())

print("Mensualités nulles :",
      (payments["payment_installments"] == 0).sum())
      


Paiements négatifs : 0
Paiements nuls : 9
Mensualités nulles : 2


### Correction des anomalies confirmées


In [48]:
items = items.drop_duplicates().copy()

display(items[
    (items["price"] <= 0)
    | (items["freight_value"] < 0)
])
df_dic["order_items"] = items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [53]:
payments = payments.drop_duplicates().copy()

display(payments[
    (payments["payment_value"] <= 0)
    | (payments["payment_installments"] <= 0)
])
display(payments["payment_type"].value_counts())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
51280,4637ca194b6387e2d538dc89b124b0ee,1,unknown,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,unknown,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
94427,c8c528189310eaa44a745b8d9d26908b,1,unknown,1,0.00


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
unknown            3
Name: count, dtype: int64

In [54]:
display(
    payments[payments["payment_type"] == "not_defined"]
    .merge(
        df_origin["orders"][["order_id", "order_status"]],
        on="order_id",
        how="left"
    )
)
payments["payment_type"] = payments["payment_type"].replace(
    "not_defined", "unknown"
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status


### Comparaison des données avant et après transformation


In [55]:
for nom in ["order_items", "order_payments"]:

    avant = df_origin[nom]
    apres = df_dic[nom]

    comparaison = pd.DataFrame({
        "Avant": [
            len(avant),
            avant.shape[1],
            avant.duplicated().sum(),
            avant.isna().sum().sum()
        ],
        "Après": [
            len(apres),
            apres.shape[1],
            apres.duplicated().sum(),
            apres.isna().sum().sum()
        ]
    }, index=[
        "Nombre de lignes",
        "Nombre de colonnes",
        "Doublons",
        "Valeurs manquantes"
    ])

    print(nom)
    display(comparaison)

order_items


,Avant,Après
Nombre de lignes,112650,112650
Nombre de colonnes,7,7
Doublons,0,0
Valeurs manquantes,0,0


order_payments


,Avant,Après
Nombre de lignes,103886,103886
Nombre de colonnes,5,5
Doublons,0,0
Valeurs manquantes,0,0
